# **Data Linkage: Data preparation and deterministic linkage**

## Introduction

This script will walk you through preparing datasets for a fictional linkage project. It will then run exact and rule-based linkage on the datasets, the latter using matchkeys in the *working_matchkeys.txt* file. 

Code in this script can be adapted for your own linkage projects.

A few tips to start with:
* For this code to work, the directory needs to be the same for all files so everything needs to be saved in the same place. Make sure the data files, matckey test file and the code are all saved in the same folder.
* This script mostly contains functions. Ensure functions are run before they are called otherwise it will cause an error message. 

The **Score-based linkage and partial agreements script** looks at how to use these methods following exact and rule-based linkage.

## Exercise

You will need to edit the working_matchkeys document to tell the linkage_exercise script which matchkeys to use, and in what order. 

**Firstly, change the working_matchkeys to achieve:**

Precision = 90% and Recall = 80%  

**And then to achieve (this second one is more difficult):**

Precision = 75% and Recall = 85%  

Remember that precision measures the correctness of the links made and recall measures the completeness of the links made. 

Then, imagining a scenario where precision and recall are equally important, which are the best matchkeys (from working_matchkey.txt file) to use, and in what order?

There is no right or wrong answer when it comes to precision and recall, the level of each depends entirely on the situation and what your users require.

*Examples of matchkey solutions to this exercise are included at the very end of this notebook.*

### Exercise tips

To improve the quality of linkage, change the matchkeys by editing the matchkey word document (for example, adding any of the derived variables). Ensure the variable name included in the matchkey document does not include the subscript in the name (e.g. ‘_a’) as this will mean the software will not recognise them as the same. 

As an example of amending this document: in the second line in the matchkey file which contains year, month and day of birth along with sex, adding the first name initial, separated by a comma reduces multiple matches, will create quite a few true positives by default. Another example could be adding one of the split postal area variables to one of the matchkey lines and see if this also increases the true positives.

When creating matchkeys consider where they will sit in your matchkey hierarchy. A hierarchical match key strategy involves using solid match keys (i.e. matchkeys with the most 'discriminating power' such as unique ID number) to start with, progressively relaxing the matchkey criteria. You can then remove variables to monitor the effects of each variable to the matchkey quality.  


# **Data preparation**

This section looks at reading in and preparing your data for exact linkage. It includes some of the steps you should also follow for preparing for rule and score-based linkage, but these also include additional steps covered later in this notebook (for rule-based) and in the subsequent script in the Data linkage learning resource (for score-based).

## Datasets

There are two datasets to prepare for linkage: **working_data_a** and **working_data_b**

 - **working_data_a** contains the variables: `id_a` (individual indentifier for this dataset), `firstname`, `middlename`, `surname`, `sex`, `dob`, `postcode' (postal area in the UK) and a record ID that is contained in the variable: `ident_a`. In addition there is a variable, 'ident_b',  that contains the record ID from the small file that it is matched to, i.e. we know the true match status.

 - **working_data_b** contains the variables: `id_b`, `firstname`, `middlename`, `surname`, `sex`, `dob`, `postcode` and a record ID that is contained in the variable: `ident_b`. In addition there is a variable, `ident_a`,  that contains the record ID from the small file that it is matched to, i.e. we know the true match status.

### Importing packages 
The code below imports packages: 

* pandas as pd (for data manipulation)
* numpy as np (a package that includes high level mathematical functions),
* re which includes regular expression support,
* os which helps interaction with the operating system (in this case it reads the working directory)
* interactiveShell, a function to ensure Jupyter notebooks run smoothly.

The last line increases the display width.

In [1]:
# Importing packages, ensuring Jupyter notebook is configured in the right way and widening the display.
import pandas as pd
import numpy as np
import re
import os

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.width', 1000)

### Get file path function
This function locates the directory (filepath) on your computer. This directory is used in subsequent functions to upload the files needed to work this notebook.

In [2]:
# This function finds the file path for this notebook and makes that the main
# directory. This is why it is important to save all files in the same folder.
def get_file_path():
    """
    Retrieves the current file path and makes this the current working directory. This function works in 
    conjunction with the read_data function. 

    Parameters:
    
    file_path : File path of working directory for this Jupyter notebook. Please note all files must be in
                this workign directory in order for this workbook to function. 

    Returns:
    
    file_path
    """
    file_path = os.getcwd()
    return file_path

In [3]:
# The below code calls the function. 
filepath = get_file_path()

## Read in data
The function below reads in the data by doing the following:
* Sets the display width to better view the data.
* Reads in the datasets to link. If you need to amend the code for your own datasets make sure you:<br>
Amend the name of the filepath, reading in a csv file and make sure all the files (including the data and matchkeys.txt)<br>
are saved in the same folder as the directory.<br>

The below code makes sure the columns within the datasets are the correct type.

<div class="alert alert-block alert-warning">
dfA = dfA.astype({"id_a":str, "firstname_a":str, "middlename_a":str, "surname_a":str,<br>
                                "sex_a":str, "dob_a":str, "postcode_a": str})
</div>
If you are amending for your own purpose make sure the variable names coincide with the variable names in your file.

The last line drops the ident variables that identifies whether the id is present in both datasets. Remember that these variables will not be present in real world examples as you will not know which ids are matches or not.

<div class="alert alert-block alert-warning">
return (dfA.drop(['ident_a','ident_b'], axis=1), dfB.drop(['ident_a','ident_b'], axis=1))<br>
</div>


In [4]:
# read_data function
def read_data(filepath):
    """
    Reads the data into the Jupyter Notebook.  

    Parameters:

    df : pandas.Dataframe
    
    dfA, dfB : The two dataframes that will be cleaned, merged and linkage metrics performed on them. 
    
    Returns:
    
    dfA, dfB
    """
    
    pd.set_option('display.width', 1000)
    
    dfA = pd.read_csv(filepath + '\\working_data_a.csv')
    dfB = pd.read_csv(filepath + '\\working_data_b.csv')
    
    dfA = dfA.astype({"id_a":str, "firstname_a":str, "middlename_a":str, "surname_a":str,
                      "sex_a":str, "dob_a":str, "postcode_a": str})
  
    dfB = dfB.astype({"id_b":str, "firstname_b":str, "middlename_b":str, "surname_b":str,
                      "sex_b":str, "dob_b":str, "postcode_b": str})
    
    return (dfA.drop(['ident_a','ident_b'], axis=1), dfB.drop(['ident_a','ident_b'], axis=1))

### dfA
Take a look at working_data_a, one of the datasets to be linked.

In [5]:
## Call the function read_data and explore dfA
dfA, dfB = read_data(filepath)
dfA.head()
print(len(dfA))

,id_a,firstname_a,middlename_a,surname_a,sex_a,postcode_a,dob_a
0,BD111HA_1,Lena,nan,Evans,2,Bd111Ha,05/03/92
1,BD111HA_10,GEORGE,LUCAS,ROBINSON,1,BD111HA,03 March 1989
2,BD111HA_100,Lucie,Lyla,Robinson,2,Bd111Ha,05 January 1985
3,BD111HA_101,Hollie,Hope,Murray,2,Bd111Ha,01-03-92
4,BD111HA_102,LUCAS GEORGE,nan,MURRAY,1,BD111HA,08-Mar-83


14934


dfA contains id_a, firstname, middlename, surname, sex, postcode, ident_b, dob, ident_a, ident_b. These are standard variables used in matching. This data is not ‘clean’ (i.e. the variables have inconsistent data inputs) which will need to be amended and standardised for the matching code to work smoothly. 

They include:
* Missingness in middle name in record BD11HA_1
* Inconsistent ways of recording sex (numeric and alphabetical)
* Inconsistent ways of recording name (including middle name in the firstname variable) in record BD111HA_102
* Missingness in the ident_b variable in record BD111HA_103
* Inconsistent ways of recording dob
* Inconsistent ways of recording postcode (with a space and without).  

All variables will need to be standardised to facilitate comparison between the two datasets to make sure we are comparing like with like.

### dfB
Then take a look at working_data_b, the other dataset to be linked.

In [6]:
## Explore dfB
dfB.head()
print(len(dfB))

,id_b,firstname_b,middlename_b,surname_b,sex_b,postcode_b,dob_b
0,BD111HA_10,GEORGE,LUCAS,ROBINSON,M,BD111HA,03/03/89
1,BD111HA_100,LUCIE,LYLA,ROBINSON,F,BD11 1HA,05-01-85
2,BD111HA_101,hollie,hope,murray,2,bd111ha,03/03/92
3,BD111HA_102,Lucas,George,Murray,1,Bd111Ha,08-Mar-83
4,BD111HA_104,george,harrison,robinson,1,bd111ha,04-03-84


7434


Similarly to dfA, there are several consistency issues with dfB:

* Inconsistent date of birth 
* Inconsistent recording of sex 
* Inconsistent use of capitals in recording name 
* Inconsistent recording of post code with upper case and lower case and a space 
* Two names contained in a name variable 
* Missingness 

## Standardising variables functions
This section demonstrates standardising variables functions for the purpose of linkage. The main reason for doing this is to ensure both datasets are able to be compared against each other. For example if a variable has a date of birth column misclassified as a string in one dataset and a datetime in the other, you will be unable to compare the two variables.

Before starting the cleaning of variables, it is a good idea to distinguish via their name the different variables in the two datasets. This is so that you always know:<br>
* what each variable represents
* know where the variable came from. This will be helpful if datasets being merged have a different methodology (e.g. gathered at a different time)
* that no variable was overwritten or misinterpreted

The belown function appends each variable in dataset 'a' with the subscript '_a' and '_b' to variables in dataset 'b'. The merge then uses these subscripts to distinguish between the datasets. 
  

In [7]:
#Add_subsctipt function
def add_subscript(letter, args):
    """
    Append a subscript suffix (e.g., '_a' or '_b') to each variable name.

    Parameters
    ----------
    letter : str
        The suffix letter to append (e.g., 'a' or 'b').
    args : List[str]
        Base variable names.

    Returns
    -------
    List[str]
        Variable names with the subscript appended.
    """
    args_subscript = []
    for arg in args:
        args_subscript.append(str(arg) + '_' + letter)
    return args_subscript

#Remove_subscript function
def remove_subscript(args):
    """
    Remove any trailing subscript (suffix after the last underscore) from each name.

    Parameters
    ----------
    args : List[str]
        Variable names possibly containing a subscript suffix.

    Returns
    -------
    List[str]
        Variable names without the trailing subscript.
    """
    args_no_subscript = []
    for arg in args:
        args_no_subscript.append(str(arg).split('_')[0])
    return args_no_subscript

### Standardising names functions: name_upper_type
This function picks out all the name variables (firstname, middlename and surname) and makes sure they are all strings and uppercase for consistency by using the *name_upper_type* function.

<div class="alert alert-block alert-danger">
Note that the add_subscript function is being called within other functions and therefore does not need to be called on its own. 
</div>

In [8]:
# Converting all the name variables to upper case
def name_upper_type(df, letter):
    """
    Convert selected name-related variables in a DataFrame to uppercase strings.

    This function takes a pandas DataFrame and a letter (used as a subscript) 
    and applies uppercase transformation to the variables:
    'firstname', 'middlename', and 'surname', after attaching the specified 
    subscript using the `add_subscript` function. Each resulting column is 
    converted to uppercase text.

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame containing name variables.

    letter : str
        A string used as a subscript or prefix when calling `add_subscript` 
        to generate the target variable names.

    Returns
    -------
    pandas.DataFrame
        The modified DataFrame with the selected name variables converted 
        to uppercase.

    """
    for name in add_subscript(letter, ["firstname", "middlename", "surname"]):
        df[name] = df[name].str.upper()
    return df

### Standardising name functions: standardise_spaces_nulls
The data contains many spaces and hyphens in the name variables which you do not want. The function below ensures that all spaces and hyphens are standardised and replaced with one space between names. It uses str.replace to override any inconsistencies within name. 

For example:

<div class="alert alert-block alert-warning">
df[name] = df[name].str.replace('-',' ').str.replace('  ',' ').str.strip()
</div>

ensures names contain no whitespace at the beginning or end by using *.str.strip()*, only have single spaces internally (by using *.str.replace('  ', ' ')* and convert all hyphens to spaces by using *.str.replace('-', ' ')*.

<div class="alert alert-block alert-warning">
np.where((df[name] == 'NAN') | (df[name] == ''), None, df[name])
</div>

The above code is conditional formatting code where if the contents of a variable is *none* or empty *(df[name] == 'NAN') | (df[name] == '')*, then replace it with *'None'* otherwise keep it's original value, in this case *df[name]*.

Although there are other methods, (e.g. *.replaace()* and *.fillna()* which will pick up anything coded as a python null (e.g. np.nan, None (in object columns), pd.NA (nullable dtypes), NaT (missing datetime)) these two functions will not pick up all variations of null in a messy datasets. It does not pick up:
* anything that hasn't been correctly coded as null (strings or empty space like *', ' ', '  ', 'NA', 'N/A', 'NULL', 'NAN'*)
* any coding specific to the dataset you are working with (e.g. *-9, ?*)

This is why nulls in this dataset have been hard coded specifically into the below function.

Note that you can adjust any of these functions for your own purposes. Remember to amend the docstring to reflect the changes so that the next person using your copy of the code can follow your changes.

In [9]:
# Getting rid of all spaces and hyphens within name variables
def standardise_spaces_nulls(df, letter):
    """
    standardise spacing and null-like values in name variables for a given subscript.

    This function targets the columns produced by:
    `add_subscript(letter, ["firstname", "middlename", "surname"])`.

    For each of those columns, it:
      - replaces hyphens (`-`) with single spaces,
      - collapses double spaces to a single space,
      - strips leading and trailing whitespace,
      - converts exact string values 'NAN' and '' (empty string) to None.

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame containing the subscripted name columns.

    letter : str
        The subscript used by `add_subscript` to resolve the target column names
        (e.g., with `letter='a'`, the function expects columns like
        'firstname_a', 'middlename_a', 'surname_a').

    Returns
    -------
    pandas.DataFrame
        The same DataFrame with standardised name columns.
    """
    for name in add_subscript(letter, ["firstname", "middlename", "surname"]):
        df[name] = df[name].str.replace('-',' ').str.replace('  ',' ').str.strip()
        df[name] = np.where((df[name] == 'NAN') | (df[name] == ''), None, df[name])
    return df

### Split names function
First name may contain a title and two names. In the below function, they are split to create three firstnames on the delimiter ' ' with the title being moved to last and deleted in the next function. 

<div class="alert alert-block alert-warning">
df[[f1, f2, f3]] = df['firstname_'+letter].str.split(' ', n=2, expand=True)
</div

The function then splits middle name and surname, ensuring each variable only contains one name.

Note that you can use the below function format in conjunction with the previous function (standardise_spaces_nulls) to split into firstname surname and middlename if all names are within one variable.  

In [10]:
def split_names(df, letter):
    """
    Split full name fields into multiple component variables based on spaces.

    This function processes the subscripted name fields:
        'firstname_<letter>', 'middlename_<letter>', and 'surname_<letter>'.

    It performs the following operations:
        - Splits the first name into up to three components:
            first1_<letter>, first2_<letter>, first3_<letter>
          using the first two spaces as delimiters.
        - Splits the middle name into up to two components:
            middle1_<letter>, middle2_<letter>
          using the first space as a delimiter.
        - Creates surname components:
            sur1_<letter>  (original surname)
            sur2_<letter>  (placeholder, initialised to None)

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame containing raw name fields. 
        Must include the columns:
            'firstname_<letter>', 'middlename_<letter>', 'surname_<letter>'.

    letter : str
        The subscript identifying the dataset source. 
        Used by `add_subscript` to generate output variable names.
        Example:
            letter='a' → first1_a, middle1_a, sur1_a, etc.

    Returns
    -------
    pandas.DataFrame
        The DataFrame with new split-name component variables added.

    """
    f1, f2, f3 = add_subscript(letter, ['first1', 'first2','first3'])
    df[[f1, f2, f3]] = df['firstname_'+letter].str.split(' ', n=2, expand=True)
    df[add_subscript(letter, ['middle1', 'middle2'])] = df['middlename_'+letter].str.split(' ', n=1, expand=True)
    df['sur1_'+letter], df['sur2_'+letter] = df['surname_'+letter], None                                                    
    return df 

### Remove titles function
This function removes any titles by putting them in a separate title column. If firstname1 is a title, swap the order of the forename variables so it becomes firstname3 (the column dropped).

<div class="alert alert-block alert-warning">
titles = ['MR', 'MRS', 'MISS', 'MS', 'DR']<br>
    df['title_' + letter] = np.where(df[f1].isin(titles), df[f1], None)
</div

The above code creates a separate variable named titles which contains *'mr', 'mrs', 'miss', 'ms', 'dr'* and creates a separate variable *'title_'* if any of the titles are in f1. This new variable will be empty if the title is not in f1 at this stage.

You then need to delete titles in f1.

<div class="alert alert-block alert-warning">
df[f1], df[f2] = np.where(<br>
        df[f1].isin(titles),<br>
        [df[f2], df[f1]],<br>
        [df[f1], df[f2]]<br>
    )
</div

To do this, the code looks for titles in f1 *df[f1].isin(titles)*, if there are any titles f1 and f2 are swapped *[df[f2], df[f1]*. If there are no titles they stay the same *[df[f1], df[f2]]*.

The same pattern is repeated for f2 and f3. Finally in the line below f3 is dropped.

<div class="alert alert-block alert-warning">
df = df.drop(columns=[f3])
</div


In [11]:
#Remove titles function
def remove_titles(df, letter):
    """  
    Identify and remove name titles (e.g., MR, MRS, MISS, MS, DR) from split 
    name components within a DataFrame.

    This function checks the variables generated by applying `add_subscript` 
    to ['first1', 'first2', 'first3'], detects whether any of these components 
    contain a title, and moves the title into a new dedicated column 
    ('title_<letter>'). The remaining name components are shifted left so that 
    first-name information appears in the earliest available position. The final 
    third component (f3) is dropped once titles have been moved.

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame containing the split name variables.

    letter : str
        A string used as a subscript to generate variable names via 
        `add_subscript`. For example, with `letter='a'`, the created variables 
        would be `first1_a`, `first2_a`, and `first3_a`.

    Returns
    -------
    pandas.DataFrame
        The modified DataFrame with:
        - extracted titles stored in 'title_<letter>',
        - name components shifted to remove title positions,
        - the third component column dropped.

    """
    f1, f2, f3 = add_subscript(letter, ['first1', 'first2','first3'])
    titles = ['MR', 'MRS', 'MISS', 'MS', 'DR']
    df['title_' + letter] = np.where(df[f1].isin(titles), df[f1], None)
    
    df[f1], df[f2] = np.where(
        df[f1].isin(titles),
        [df[f2], df[f1]],
        [df[f1], df[f2]]
    )

    df[f2], df[f3] = np.where(
        df[f2].isin(titles),
        [df[f3], df[f2]],
        [df[f2], df[f3]]
    )

    df = df.drop(columns=[f3])

    return df


### Converting missing postal area, standardising spaces and converting to upper case
This function standardises postal area by deleting extra spaces, converts postcode to upper case, replaces missing postcodes with ‘None’ and converts all inputs into strings. pc is shorthand for postcode (postal area in the UK).

The different variations of empty have been explicitly coded into the function below:

<div class="alert alert-block alert-warning">
df[pc] = np.where((df[pc] == 'NAN') | (df[pc] == '') | (df[pc].isnull()), None, df[pc])
</div

Here nones can be NAN or blank space or correctly coded as *.isnull()*. If any of the entries contain these they are recoded correctly as none otherwise they are left as is.

The reason for this is due to variations in working with addministrative data as:
* Administrative data can include inputs that are incorrect or misleading. For nulls it can just be empty, include an unauthorised token or correctly coded as none.
* Nones can be coded within python differently depending on the variable type. 'NAN', '', None, NaN, pd.NA are all different ways of coding a null. For linkage, all null variables need to be coded exactly the same to ensure consistency as different nulls are treated differently by Python when doing a merge.  

In [12]:
# Missing postal area
def upper_missing_postal_area(df, pc):
    """
    standardise a postal-area column by uppercasing values, removing spaces,
    and converting null-like strings to proper missing values.

    This function cleans a specified postal-area (or postcode) column by:
      - stripping all spaces,
      - converting all characters to uppercase,
      - replacing the literal strings 'NAN' and '' (empty string) with None,
      - converting actual missing values (NaN) to None,
      - returning the column as a string dtype.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the postal-area column to clean.

    pc : str
        The name of the column containing postcode or postal-area values.

    Returns
    -------
    pandas.DataFrame
        The DataFrame with the cleaned and standardised postal-area column.

    """
    df[pc] = df[pc].str.replace(' ','').str.upper()
    df[pc] = np.where((df[pc] == 'NAN') | (df[pc] == '') | (df[pc].isnull()), None, df[pc])
    return df.astype({pc:str})

### Sex upper function
The below function ensures that sex is upper case.

In [13]:
# Sex upper
def sex_upper(df, sex):
    """
    Convert the values of a sex/gender variable to uppercase.

    This function takes the name of a column containing sex or gender codes 
    (e.g., 'm', 'f', 'male', 'Female') and converts all values in that column 
    to uppercase text. This is often used to standardise categorical variables 
    prior to harmonisation or linkage.

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame containing the sex variable to be standardised.

    sex : str
        The name of the column in `df` that stores sex or gender values.

    Returns
    -------
    pandas.DataFrame
        The DataFrame with the specified sex column converted to uppercase.

    """
    df[sex] = df[sex].str.upper()
    return df

### Standardise sex function
The below function ensures that any potential variation of input into the sex column is converted into 1 and 2. As an example:

<div class="alert alert-block alert-warning">
df.loc[(df[sex] == 'M') | (df[sex] == 'MALE'), sex] = 1
</div

*.loc* function is used to modify the sex variable whereby if it equals M or MALE then it is changed to 1.



In [14]:
# Standardise sex
def standardise_sex(df, sex):
    """
    Standardise a sex variable by recoding common text labels into numeric codes.

    This function converts the values in a specified sex/gender column so that:
        - 'M' or 'MALE'   → 1
        - 'F' or 'FEMALE' → 2

    The final column is returned as a float dtype, following common coding 
    conventions in linkage and statistical processing pipelines where numeric 
    categorical representations are required.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the sex variable to be standardised.

    sex : str
        The name of the column in `df` containing sex or gender values. 
        Values are expected to already be uppercased prior to calling this 
        function (e.g., via `sex_upper`).

    Returns
    -------
    pandas.DataFrame
        The DataFrame with the specified sex column recoded to numeric values 
        (1.0 for male, 2.0 for female).

    """

    df.loc[(df[sex] == 'M') | (df[sex] == 'MALE'), sex] = 1
    df.loc[(df[sex] == 'F') | (df[sex] == 'FEMALE'), sex] = 2
    return df.astype({sex:float})
    

### Standardise date of birth
The below function ensures that any input into the date of birth field is consistent. 

In [15]:
#Convert dob to a datetime variable
def standardise_dob(df, dob):
    """
    Standardise a date‑of‑birth column by converting it to a pandas datetime dtype.

    This function converts the specified date‑of‑birth column into a 
    `datetime64[ns]` dtype using `pandas.to_datetime`. It interprets dates using 
    day-first format (DD/MM/YYYY), which is the standard format for UK dates and 
    many administrative datasets.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the date-of-birth field to be converted.

    dob : str
        The name of the column in `df` that contains date-of-birth values.

    Returns
    -------
    pandas.DataFrame
        The DataFrame with the date-of-birth column converted to a datetime type.
    """
    df[dob] = pd.to_datetime(df[dob], dayfirst=True)
    return df

### Clean data function
This function calls all the functions in this section that standardise and clean the variables used for data matching. Standardising variables is crucial for any matching strategy to ensure like with like comparisons are made. These functions standardise the linkage variables for the name variables, postcode, sex and date of birth. .



In [16]:
# Main standardisation function which calls all previous functions
def clean_data(df, letter):
    """
    Run the full name, address, sex, and date-of-birth standardisation pipeline 
    for a specific dataset subscript.

    This function sequentially applies the standardisation and cleaning functions 
    used across the linkage pipeline. It operates on a set of variables that use 
    a shared subscript (e.g., 'firstname_a', 'postcode_a', 'sex_a'), ensuring 
    consistent formatting prior to matching or deterministic linkage.

    The steps performed are:
        1. Convert name fields to uppercase (`name_upper_type`).
        2. Standardise spacing, hyphens, and null-like values in name fields 
           (`standardise_spaces_nulls`).
        3. Split multi-part names into component fields (`split_names`).
        4. Identify and remove titles (e.g., MR, MRS) from name components 
           (`remove_titles`).
        5. Clean postal-area or postcode information by uppercasing and handling 
           missing values (`upper_missing_postal_area`).
        6. Uppercase the sex variable (`sex_upper`).
        7. Standardise the sex variable into numeric codes (`standardise_sex`).
        8. Standardise the date-of-birth field (`standardise_dob`).

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame containing the subscripted variables to be cleaned.

    letter : str
        The dataset identifier used as a subscript in variable names 
        (e.g., 'a' → 'firstname_a', 'sex_a', 'dob_a').

    Returns
    -------
    pandas.DataFrame
        The cleaned DataFrame with all relevant fields standardised and 
        transformed according to the pipeline.

    """    
    df = name_upper_type(df, letter)
    df = standardise_spaces_nulls(df, letter)
    df = split_names(df, letter)
    df = remove_titles(df, letter)
    
    df = upper_missing_postal_area(df, 'postcode_' + letter)
    
    df = sex_upper(df, 'sex_' + letter)
    df = standardise_sex(df, 'sex_' + letter)
    
    df = standardise_dob(df, 'dob_' + letter)
    return df


In [17]:
### Calling this function
dfA = clean_data(dfA, "a")
dfB = clean_data(dfB, "b")

C:\Users\bestj\AppData\Local\Temp\ipykernel_32372\827209228.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[dob] = pd.to_datetime(df[dob], dayfirst=True)
C:\Users\bestj\AppData\Local\Temp\ipykernel_32372\827209228.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[dob] = pd.to_datetime(df[dob], dayfirst=True)


Take a look at the data now that it has been prepared.

In [18]:
dfA.head()

,id_a,firstname_a,middlename_a,surname_a,sex_a,postcode_a,dob_a,first1_a,first2_a,middle1_a,middle2_a,sur1_a,sur2_a,title_a
0,BD111HA_1,LENA,None,EVANS,2.0,BD111HA,1992-03-05,LENA,None,None,None,EVANS,None,None
1,BD111HA_10,GEORGE,LUCAS,ROBINSON,1.0,BD111HA,1989-03-03,GEORGE,None,LUCAS,None,ROBINSON,None,None
2,BD111HA_100,LUCIE,LYLA,ROBINSON,2.0,BD111HA,1985-01-05,LUCIE,None,LYLA,None,ROBINSON,None,None
3,BD111HA_101,HOLLIE,HOPE,MURRAY,2.0,BD111HA,1992-03-01,HOLLIE,None,HOPE,None,MURRAY,None,None
4,BD111HA_102,LUCAS GEORGE,None,MURRAY,1.0,BD111HA,1983-03-08,LUCAS,GEORGE,None,None,MURRAY,None,None


In [19]:
dfB.head()

,id_b,firstname_b,middlename_b,surname_b,sex_b,postcode_b,dob_b,first1_b,first2_b,middle1_b,middle2_b,sur1_b,sur2_b,title_b
0,BD111HA_10,GEORGE,LUCAS,ROBINSON,1.0,BD111HA,1989-03-03,GEORGE,None,LUCAS,None,ROBINSON,None,None
1,BD111HA_100,LUCIE,LYLA,ROBINSON,2.0,BD111HA,1985-01-05,LUCIE,None,LYLA,None,ROBINSON,None,None
2,BD111HA_101,HOLLIE,HOPE,MURRAY,2.0,BD111HA,1992-03-03,HOLLIE,None,HOPE,None,MURRAY,None,None
3,BD111HA_102,LUCAS,GEORGE,MURRAY,1.0,BD111HA,1983-03-08,LUCAS,None,GEORGE,None,MURRAY,None,None
4,BD111HA_104,GEORGE,HARRISON,ROBINSON,1.0,BD111HA,1984-03-04,GEORGE,None,HARRISON,None,ROBINSON,None,None


Now that you have standardised and cleaned your datasets, they are ready for exact linkage. 

There is additional preparation you can do if you want to run rule-based linkage. These steps are covered later.

# **Exact matching**

Now that you have cleaned and standardised the variables, you can perform exact linkage by using the *merge* function. Below is a function that does this for you. 

As a reminder, the advantages of exact linkage are:
* Exact linkage can be very efficient. It has the potential to pick up a large number of links (true matches) in a very short space of time.
* Because of its simplicity, exact linkage is very easy to implement. It does not require much effort to code at all - exact linkage happens by default when running a simple join or merge. 
* Only accepting exact matches as links has its drawbacks, but it also minimises the risk of false positives. If two records match exactly, it is incredibly unlikely that they do not refer to the same person!

The disadvantages are:
* Exact linkage does not allow for any kind of discrepancies between values, even if to a human these might be obvious. A name could be shortened to an easily recognisable nickname, or a single letter or a number could be out of place. Despite these kinds of discrepancies being very common in administrative data, exact linkage would still consider these non-matches.
* Subsequently, if used on its own, exact linkage can leave a large number of missed links (false negatives).

You have an individual identifier in the two datasets which can be used to match also. Note that after this step the individual identifier is not used to show you how linkage would work without a variable in common to identify records. For this dataset, the individual identifier is used as a *'source of truth'*. How to use these variables will be explained and demonstrated further on in this notebook.

The below function merges the two datasets using all available variables. It uses a *key_map* to outline specifically which variables should be matched. This is useful when the two datasets do not have the same names variables. 

In [20]:
# Exact_link_with_key_mapping_function
def exact_link_with_key_mapping(left, right, key_map, how, indicator):
    """
    Run exact linkage by merging two DataFrames whose join keys have different names.

    Parameters
    ----------
    left, right : pd.DataFrame
    key_map : dict
        Mapping from left key name -> right key name.
        Supports multiple keys, e.g. {'Age': 'age_years', 'DOB': 'birth_date'}
    how : str
        'left', 'right', 'inner', 'outer'
     indicator : bool
         if True, adfd pandas merge indicator column '_merge'.
         
    Returns
    -------
    pd.DataFrame
    """
    left_on = list(key_map.keys())
    right_on = [key_map[k] for k in left_on]

   
    merged = pd.merge(
        left, right,
        how=how,
        left_on=left_on,
        right_on=right_on,
        indicator=indicator
    )

    return merged

In [21]:
# Run exact_link_with_key_mapping function
key_map = {
    "id_a": "id_b",
    "firstname_a": "firstname_b",
    "middlename_a": "middlename_b",
    "surname_a": "surname_b",
    "sex_a": "sex_b",
    "postcode_a": "postcode_b",
    "dob_a": "dob_b"
}

linked_df = exact_link_with_key_mapping(
    dfA,
    dfB,
    key_map=key_map,
    how="inner",
    indicator=True
)

In [22]:
# Have a look at the number of matched records
print('Number of matched records in dataframe: ',len(linked_df))

Number of matched records in dataframe:  4474


In [23]:
# Run exact_link_with_key_mapping for residuals (those that did not link) and show the residuals
residuals = exact_link_with_key_mapping(dfA, dfB, key_map, how="outer", indicator=True)
residualsA = residuals[residuals['_merge'] == 'left_only'] 
print('Number of unmatched records in dataframeA: ',len(residualsA))

residualsB = residuals[residuals['_merge'] == 'right_only'] 
print('Number of unmatched records in dataframeB: ',len(residualsB))

Number of unmatched records in dataframeA:  10460
Number of unmatched records in dataframeB:  2960


There are 2960 unmatched records in dfB and 10460 unmatched records in dfA. This makes sense as dfA is signicantly larger than dfB. 

# **Rule-based linkage**

Running rule-based linkage after exact linkage helps you to identify more matches. However, there are some additional steps to preparing your datasets so that your rule-based linkage works as intended. These are covered below.

As a reminder, the advantages of rule-based linkage are:

* Through the use of multiple match keys, rule-based linkage allows you to account for many different types of inconsistencies and errors in your datasets. Real datasets, especially administrative datasets, are prone to these, and using exact linkage alone to link pairs between them would be very limiting.
* Match keys are highly customisable. This gives you flexibility and allows you to optimise linkage to your specific needs and targets.
* Though less so than exact linkage, rule-based linkage is efficient and not very computationally expensive to run. 

The disadvantages are:

* Rule-based linkage introduces more false positives than exact linkage. This is especially the case if your match keys are poorly designed.
* In a real project where you cannot necessarily confirm which matches are true positives, it is tricky to determine how good your match keys and rules are.
* Because it requires more thought and effort, especially when experimenting with different match keys, rule-based linkage takes more time than exact linkage.


## Parsing and deriving variables
This section relaxes certain variables which can be added to matchkeys. This process can be useful to increase matches due to errors in the data. Typically, they would be added to *hierarchical match keys* which involve having the strictest match keys first, usually the exact match keys, and then match keys are gradually loosened to allow more variation/error. 

Some examples of producing relaxed variables can be seen where initials are derived from names to increase linkage between two records in the case of a spelling mistake or nickname use.  

<div class="alert alert-block alert-danger">
Note that this may increase the false positives as you are matching on a section of a variable rather than the full variable.
</div>   

### Parse_short_names function
This function first appends the subscript *"_a" or "_b"* to created name variables . It then parses the first three letters of each name into a new variable which includes 'short' at the beginning of the old variable (e.g. shortfirst1). It then does the same for 'init', including the first letter of a name variable in the column named for example initfirst1. 

<div class="alert alert-block alert-warning">
f1, f2, m1, m2, s1, s2 = add_subscript(letter, ['first1', 'first2', 'middle1', 'middle2', 'sur1', 'sur2'])<br>
    for name in [f1, f2, m1, m2, s1, s2]:<br>
        df['short'+name] = df[name].str[:3]<br>
        df['init'+name] = df[name].str[0]'
</div>

Then it combines all of the initial columns to give a person's initials.

<div class="alert alert-block alert-warning">
df['initials_'+letter] = df[['init'+f1,'init'+f2,'init'+m1,'init'+m2,'init'+s1,'init'+s2]].apply(<br>
            lambda row: row.str.cat(sep=''), axis=1)<br>
</div><br>


<div class="alert alert-block alert-danger">
Note that if date of birth and initials are common with no unique identifier or address variable this technique is likely to create significantly more false negative matches.
</div>


In [24]:
#parse short names function
def parse_short_names(df, letter):
    """
    Generate shortened name variants and initials for split name components.

    This function creates two new sets of variables for each name component 
    associated with the provided subscript (`letter`):

        - **short<col>**: contains the first three characters of the name component.
        - **init<col>**: contains the first character (initial) of the name component.

    It also constructs a composite initials field combining all initials in order:
        initials_<letter>

    The function operates on the following subscripted name components, created 
    using `add_subscript`:
        - first1_<letter>, first2_<letter>
        - middle1_<letter>, middle2_<letter>
        - sur1_<letter>, sur2_<letter>

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the split name components to be transformed.

    letter : str
        The subscript used to identify the dataset-specific name variables.
        Example: with `letter = 'a'`, the function expects:
            first1_a, first2_a, middle1_a, middle2_a, sur1_a, sur2_a

    Returns
    -------
    pandas.DataFrame
        The DataFrame with additional short-name, initial, and combined initial
        variables.

    """
    f1, f2, m1, m2, s1, s2 = add_subscript(letter, ['first1', 'first2', 'middle1', 'middle2', 'sur1', 'sur2'])
    for name in [f1, f2, m1, m2, s1, s2]:
        df['short'+name] = df[name].str[:3]
        df['init'+name] = df[name].str[0]
    
    df['initials_'+letter] = df[['init'+f1,'init'+f2,'init'+m1,'init'+m2,'init'+s1,'init'+s2]].apply(
            lambda row: row.str.cat(sep=''), axis=1)
    return df


### Parse_postcode (postal area) function
The below function parses or splits the postcode (postal area) into area and district (or region). As you will remember from the main guidance, this is helpful when doing rule-based matching as it gives the ability to link records on their area only. There could also be a hierarchy of asking the software to match the whole postcode as well as matching the district only due to typos. 

The function adds the subscript to the relevant variables, then extracts the section of the postcode that is the area and sector. An example of postcode area in the UK is: the first one to two letters followed by a number at the beginning of a postcode. An example of a postcode district in the UK would be RD (the city of Reading) or SW (southwest London). The code below extracts district from the postcodde variable.  

<div class="alert alert-block alert-warning">
df[area] = df[pc].str.extract('\\A([A-Z]{1,2})', expand=True)
</div>

This sequence means extract this sequence if seen in the pc variable: at the beginning of the string *\\A* look for one to two letters *([A-Z]{1,2}* and input this into the new pcarea variable.

The second sequence searches for the district, this sequence is the full section of the first part of a UK postcode (e.g. SW15, RD1)

<div class="alert alert-block alert-warning">
df[district] = df[pc].str.extract('\\A([A-Z]{1,2}[0-9]{1,2})[0-9]', expand=True)
</div>

This sequence means extract this sequence if seen in the pc variable: at the beginning of the string *\\A* one to two letters *([A-Z]{1,2}* followed by one to two numbers *[0-9]{1,2}* followed by a number *[0-9]* and input this into the pcdistrict variable if found.   

The final sequence looks for the full area sequence (e.g. RD1 for Reading, SW15 for Southwest London or B5 for Birmingham).  

<div class="alert alert-block alert-warning">
np.where((df[district].isnull()) & (df[pc].str.match('[A-Z]{1,2}[0-9]{1,2}')),<br>
                              df[pc], df[district])
</div>

This is fallback code if previous searches above have not worked. If pcdistrict is missing but the postcode looks like a valid outward code (area and district), then replace pcdistrict with the outward code. Otherwise keep whatever is already in pcdistrict.

Evidence shows that when moving in the UK, most will stay within the same area or even district. This is useful to know if the dataset has not been updated in a timely way as records can still be matched. 

In [25]:
#parse_postcode function
def parse_postcode(df, letter):
    """
    Parse a postcode into its outward components: area and district.

    This function extracts two standard UK postcode elements from the
    subscripted postcode variable:

        - **pcarea_<letter>**: the postcode area (first 1–2 letters),
          e.g. 'B', 'SW', 'RG'.
        - **pcdistrict_<letter>**: the postcode district (area + 1–2 digits),
          e.g. 'B5', 'SW15', 'RG2'.

    It uses regular expressions to extract these components and includes
    a fallback rule for district extraction when the standard pattern does
    not capture cases such as single‑digit districts.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the postcode field to be split.

    letter : str
        The subscript identifying the dataset-specific postcode column.
        For example, with `letter='a'` the function expects a column:
            'postcode_a'
        and will create:
            'pcarea_a', 'pcdistrict_a'.

    Returns
    -------
    pandas.DataFrame
        The DataFrame with added postcode area and district columns.

    """
    pc, area, district = add_subscript(letter, ['postcode', 'pcarea', 'pcdistrict'])
    df[area] = df[pc].str.extract('\\A([A-Z]{1,2})', expand=True)
    df[district] = df[pc].str.extract('\\A([A-Z]{1,2}[0-9]{1,2})[0-9]', expand=True) 
    
    df[district] = np.where((df[district].isnull()) & (df[pc].str.match('[A-Z]{1,2}[0-9]{1,2}')),
                              df[pc], df[district])
    return df

### Parse variables function
In data linkage it is good practice to parse date variables into day, month, and year to ensure consistency between datasets and link on these individually if needed. It is also good practice after creating these three separate variables to check for unlikely inputs (e.g. 32/01/1986). 

<div class="alert alert-block alert-danger">
Note that the short_names function and split_postcode are called within the create_derived_variables function therefore ensure you call these functions before you call the create_derived_variables function.
</div>

In [26]:
#Main parse_variables function
def parse_variables(df, letter):
    """
    Create derived variables for names and date of birth, and parse the postcode.

    This function:
      1) generates short-name and initial variables via `parse_short_names(df, letter)`,
      2) derives day, month, and year of birth from the subscripted DOB column,
      3) parses the subscripted postcode into area and district via `parse_postcode(df, letter)`.

    Specifically, it expects a date-of-birth column named:
        'dob_<letter>'
    and creates:
        - 'daybirth_<letter>'   (integer day)
        - 'monthbirth_<letter>' (integer month)
        - 'yearbirth_<letter>'  (integer year)

    It also relies on the existence of a postcode column:
        'postcode_<letter>'
    which is split by `parse_postcode` into:
        - 'pcarea_<letter>'     (postcode area, e.g., 'B', 'SW', 'RG')
        - 'pcdistrict_<letter>' (postcode district, e.g., 'B5', 'SW15', 'RG2')

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing subscripted name, DOB, and postcode fields.

    letter : str
        The dataset subscript used to resolve column names (e.g., 'a' → 'dob_a',
        'postcode_a', etc.).

    Returns
    -------
    pandas.DataFrame
        The DataFrame with:
          - short-name and initial columns added (from `parse_short_names`),
          - derived day/month/year of birth columns,
          - postcode area and district columns added (from `parse_postcode`).

    """
    df = parse_short_names(df, letter)
    
    dob = 'dob_' + letter
    df['daybirth_'+letter] = df[dob].dt.day
    df['monthbirth_'+letter] = df[dob].dt.month
    df['yearbirth_'+letter] = df[dob].dt.year
    
    return parse_postcode(df,letter)


In [27]:
# Call this function 
dfA = parse_variables(dfA, "a")
dfB = parse_variables(dfB, "b")

In [28]:
# Have a look at the data
dfA.head()

,id_a,firstname_a,middlename_a,surname_a,sex_a,postcode_a,dob_a,first1_a,first2_a,middle1_a,...,shortsur1_a,initsur1_a,shortsur2_a,initsur2_a,initials_a,daybirth_a,monthbirth_a,yearbirth_a,pcarea_a,pcdistrict_a
0,BD111HA_1,LENA,None,EVANS,2.0,BD111HA,1992-03-05,LENA,None,None,...,EVA,E,None,None,LE,5.0,3.0,1992.0,BD,BD11
1,BD111HA_10,GEORGE,LUCAS,ROBINSON,1.0,BD111HA,1989-03-03,GEORGE,None,LUCAS,...,ROB,R,None,None,GLR,3.0,3.0,1989.0,BD,BD11
2,BD111HA_100,LUCIE,LYLA,ROBINSON,2.0,BD111HA,1985-01-05,LUCIE,None,LYLA,...,ROB,R,None,None,LLR,5.0,1.0,1985.0,BD,BD11
3,BD111HA_101,HOLLIE,HOPE,MURRAY,2.0,BD111HA,1992-03-01,HOLLIE,None,HOPE,...,MUR,M,None,None,HHM,1.0,3.0,1992.0,BD,BD11
4,BD111HA_102,LUCAS GEORGE,None,MURRAY,1.0,BD111HA,1983-03-08,LUCAS,GEORGE,None,...,MUR,M,None,None,LGM,8.0,3.0,1983.0,BD,BD11


### Checking date of bith
As previously mentioned, it is a good idea to check date of birth (once standardised). 

In [29]:
#check for invalid inputs for date of birth in dfA
dfA[["daybirth_a", "monthbirth_a", "yearbirth_a"]].describe()

,daybirth_a,monthbirth_a,yearbirth_a
count,14913.000000,14913.000000,14913.000000
mean,4.483136,2.015892,1987.981828
std,2.319851,0.835410,3.139890
min,1.000000,1.000000,1983.000000
25%,2.000000,1.000000,1985.000000
50%,4.000000,2.000000,1988.000000
75%,7.000000,3.000000,1991.000000
max,10.000000,8.000000,1993.000000


In [30]:
#Check for invalid values in dfB
dfB[["daybirth_b", "monthbirth_b", "yearbirth_b"]].describe()

,daybirth_b,monthbirth_b,yearbirth_b
count,7429.000000,7429.000000,7429.000000
mean,4.455512,2.023018,1988.034729
std,2.325161,0.844465,3.113216
min,1.000000,1.000000,1983.000000
25%,2.000000,1.000000,1985.000000
50%,4.000000,2.000000,1988.000000
75%,7.000000,3.000000,1991.000000
max,10.000000,8.000000,1993.000000


There are no unexpected inputs.

### Deriving age
As date of birth is static, it is standard practice to match on this rather than age (as this changes year on  year). However, if despite this age is needed for other reasons (e.g. quality assuring the matches by checking for patterns in the data), the below code to derive age is outlined. It is then added to another dataset *dfA_check* and *dfB_check* as, if this is added to the linkage dataset, exact matching will become too rigid.

Ensure whether the age at analysis is relevant or the age at which the data was collected. Are these the same in both datasets?  


In [31]:
# Derives age from date of birth
def age(df, letter):
    """
   Derive age (in whole years) from a subscripted date-of-birth column.

    This function computes age as of "today" (based on the local system date),
    using the column name pattern `dob_<letter>`, and writes the result to
    `age_<letter>`. The calculation:
      - takes the difference in calendar years between today and the DOB year,
      - subtracts 1 if the birthday has not yet occurred this year,
      - sets age to None where DOB is missing or unparseable.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the date-of-birth column to use.

    letter : str
        The subscript identifying the dataset-specific DOB column name.
        For example, with `letter='a'`, the function expects a column
        named `dob_a` and will create `age_a`.

    Returns
    -------
    pandas.DataFrame
        The DataFrame with a new column `age_<letter>` containing integer-like
        ages (dtype may be float to accommodate missing values).

    """
    
    dob_col = f"dob_{letter}"
    age_col = f"age_{letter}"
    dob = pd.to_datetime(df[dob_col], errors="coerce")
    today = pd.Timestamp.today().normalize()
    years = today.year - dob.dt.year
    
    before_birthday = (
        (today.month < dob.dt.month) |
        ((today.month == dob.dt.month) & (today.day < dob.dt.day))
    )

    df[age_col] = years - before_birthday
    df.loc[dob.isna(), age_col] = None
    return df

In [32]:
# Call the function on different datasets 
dfA_check = age(dfA, "a")
dfB_check = age(dfB, "b")

dfB_check.head()

,id_b,firstname_b,middlename_b,surname_b,sex_b,postcode_b,dob_b,first1_b,first2_b,middle1_b,...,initsur1_b,shortsur2_b,initsur2_b,initials_b,daybirth_b,monthbirth_b,yearbirth_b,pcarea_b,pcdistrict_b,age_b
0,BD111HA_10,GEORGE,LUCAS,ROBINSON,1.0,BD111HA,1989-03-03,GEORGE,None,LUCAS,...,R,None,None,GLR,3.0,3.0,1989.0,BD,BD11,37.0
1,BD111HA_100,LUCIE,LYLA,ROBINSON,2.0,BD111HA,1985-01-05,LUCIE,None,LYLA,...,R,None,None,LLR,5.0,1.0,1985.0,BD,BD11,41.0
2,BD111HA_101,HOLLIE,HOPE,MURRAY,2.0,BD111HA,1992-03-03,HOLLIE,None,HOPE,...,M,None,None,HHM,3.0,3.0,1992.0,BD,BD11,34.0
3,BD111HA_102,LUCAS,GEORGE,MURRAY,1.0,BD111HA,1983-03-08,LUCAS,None,GEORGE,...,M,None,None,LGM,8.0,3.0,1983.0,BD,BD11,43.0
4,BD111HA_104,GEORGE,HARRISON,ROBINSON,1.0,BD111HA,1984-03-04,GEORGE,None,HARRISON,...,R,None,None,GHR,4.0,3.0,1984.0,BD,BD11,42.0


### Cleaning real world administrative data
Included in this script is a dummy dataset which does not include any missing values. Remember that in real data, there will be nulls which you will need to think about how to deal with. Bear in mind that missingness does not mean that there is no match. 

You may also need to remove out of scope records (e.g. unrealistic birthdays as you have searched for previously here or deceased records). Finally, be careful of duplicate records in your data so check for this before any matching occurs to ensure it does not create more matches than there actually are.  

## Create matchkeys function
The functions below demonstrate exact matching (a type of rule based matching) using the matchkeys file. This file contains the variables to match on in linkage. As previously mentioned, it can be updated and re-run if any matchkeys are added or removed from the matchkeys file. Remember that a matching variable needs to have:
* Low level of missing values
* Low level of error
* High distinguishing power
* Unchanging

In the field, start with a standard set of keys and clerically review samples of each to estimate their false
positive rate. You could then relax some of the keys and create hierarchical matchkeys. Other methods for creating high quality mathckeys include:
* Remove or tighten particularly low-quality keys
* If possible, review missed matches & design keys to capture these
* Tip: Do not try to exhaust all the possibilities as you might end up with lots of false positives

This function reads the matchkeys file into the jupyter session. It then allows multiple matches *(multiple_matches= True)* otherwise known as one to many matches or does not allow multiple matches *(multiple_matches = False)* otherwise known as one to one matches.

It reads the text file into the session as a list of lines using this code, storing them as a list. 

<div class="alert alert-block alert-warning">
lines = file.read().splitlines()<br>
    file.close()<br>
    matchkeys = []
</div>

The default of whether to have multiple matches will be set to false.

<div class="alert alert-block alert-warning">
keep_multiple = False
</div>

There is then a loop function to identify the line 'include multiple matches'. If after this sentence there is a true, the software will allow multiple matches (one to many). If these are input incorrectly into the file, there will be an error message 

<div class="alert alert-block alert-warning">
'must be either "true" or "false" but received "' + value + '"'
</div>

It will do nothing if there is a blank space and will append the matchkey to the list of matchkeys. This function will produce the change in keep_multiple and the updated matchkeys.

For the matching exercise, amend this file and run the code from here.

In [33]:
# Downloading matchkey file into notebook
def create_matchkeys(filepath):
    """
    Parse a matchkeys configuration file and return matching rules and policy.

    This function reads a text configuration file located at:
        <filepath>/working_matchkeys.txt

    The file is expected to contain:
      - A control line specifying whether to include multiple matches:
            Include multiple matches = true|false
        (case-insensitive; whitespace around '=' is allowed)
      - One or more non-empty lines defining matchkeys, one per line.
        Each matchkey line is a comma-separated list of variable names
        that constitute a single matching rule. Whitespace is ignored.

    Example file contents
    ---------------------
    Include multiple matches = TRUE
    firstname_a, surname_a, dob_a
    postcode_a, dob_a
    initials_a, yearbirth_a

    The function returns:
      - `keep_multiple`: a boolean indicating whether multiple matches should be
        retained (True) or filtered out (False).
      - `matchkeys`: a list of lists, where each inner list is a matchkey rule,
        e.g., [['firstname_a', 'surname_a', 'dob_a'], ['postcode_a', 'dob_a'], ...].

    Parameters
    ----------
    filepath : str
        Directory path containing the configuration file
        'working_matchkeys.txt'.

    Returns
    -------
    tuple[bool, list[list[str]]]
        A 2-tuple `(keep_multiple, matchkeys)` where:
          - keep_multiple : bool
          - matchkeys : list of matchkey rules (each rule is a list of column names)

    Raises
    ------
    ValueError
        If the control line for "Include multiple matches" is present but does
        not specify 'true' or 'false'.
    FileNotFoundError
        If '<filepath>/working_matchkeys.txt' cannot be found (propagated from `open`).
    """
    file = open(filepath + '/working_matchkeys.txt')
    lines = file.read().splitlines()
    file.close()
    matchkeys = []
    keep_multiple = False
    
    for line in lines:
        if re.match('Include multiple matches', line):
            value = re.search('Include multiple matches *=(.*)', line).group(1).replace(' ','')
            if value.upper() == 'TRUE':
                keep_multiple = True
            elif value.replace(' ','').upper() == 'FALSE':
                keep_multiple = False
            else:
                raise ValueError('"Include multiple matches" in the file matchkeys.txt '+\
                                 'must be either "true" or "false" but received "' + value + '"')
        elif line == '':
            continue
        else:
            matchkeys.append(line.replace(' ','').split(','))
    return keep_multiple, matchkeys

In [34]:
# Call the working_matchkeys function
keep_multiple, matchkeys = create_matchkeys(filepath)

### Run rule-based matching function 
The below code starts by using matchkeys to perform rule-based linkage. Remember that:
* Each stage of rule-based linkage is done sequentially after exact linkage
* Order matchkeys on strength of finding matches
* Relax rules gradually and systematically
* Run clerical checks
* Check number of links made at each stage
* Earlier matchkeys over-rule later ones.

As described previously, deriving certain variables such as district and area from postcodes and including this in any matching strategy can increase the match rate. Some more examples include: excluding the sex variable if it contains missing data, or only including a district variable if the population of focus has a high rate of mobility and is therefore likely to have moved if the data collection dates are different between the two datasets. Note that the rules need to not be so relaxed to increase false matches. If you would like to include some of the derived variables in your matching strategy, the matchkeys file will need to be amended and uploaded again into the notebook by running the *create_matchkeys* function again.

The below code first of all adds the subscript _a and _b to the matchkey variables. It then joins them using an inner merge to show only the records that merge in the final dataset.

It then removes all matches but one if keep_multiple = False. It does this by counting the number of matches and if the number is above one, it removes all other record matches. 

<div class="alert alert-block alert-warning">
linked['multi_match'] = linked['id_a'].map(linked['id_a'].value_counts()>1)\<br>
                                | linked['id_b'].map(linked['id_b'].value_counts()>1)<br>
        multiple_matches = len(linked[linked['multi_match']])<br>
        linked = linked[linked['multi_match'] == False].drop('multi_match', axis=1)
</div>

<div class="alert alert-block alert-danger">
Note that if you are linking one to many, many to many or many to one, do not use the above code as you want to keep and analyse multiple matches. You will also need to amend the matchkeys file to keep multiple matches = TRUE.  
</div>

In order to quantify the matches, it adds a match_status column to the dataset. If there is a match for the record, the match_status is equal to 1, if not it is equal to 0. 

<div class="alert alert-block alert-danger">
if output_path is not None:<br>
        os.makedirs(output_path, exist_ok=True)<br>
        key_name = "_".join(matchkey)<br>
        filename = f"linked_{key_name}.csv"<br>
        full_path = os.path.join(output_path, filename)<br>
        linked.convert_dtypes().to_csv(full_path, index=False, encoding="utf-8-sig")
</div>

The final section of this function creates a file to export the matches. It first checks whether the filepath exists, gives it a name, then gives it the final path where the file should be saved. Finally it makes sure all data types are consistent with turning it into a csv.


In [35]:
# Rule_based_linkage_function
def rule_based_linkage_via_matchkeys(dfA, dfB, matchkey, keep_multiple, output_path=None):
    """
    Merge dfA and dfB on subscripted `matchkey`, optionally filter multiple matches,
    and (optionally) export the linked dataset to CSV.

    This function expects that `dfA` contains a column matching `add_subscript('a', matchkey)`
    and `dfB` contains a column matching `add_subscript('b', matchkey)`. It performs an inner
    merge on those keys, optionally removes rows that are part of multiple matches on either
    side, and labels exact ID matches.

    Parameters
    ----------
    dfA : pd.DataFrame
        Left DataFrame. Must contain columns: `add_subscript('a', matchkey)` and `id_a`.
    dfB : pd.DataFrame
        Right DataFrame. Must contain columns: `add_subscript('b', matchkey)` and `id_b`.
    matchkey : str
        The base column name to be transformed by `add_subscript` for the merge keys.
    keep_multiple : bool
        If False, rows where `id_a` or `id_b` occurs more than once in the merged result
        are removed. If True, multiple matches are retained.

    Returns
    -------
    linked : pd.DataFrame
        The merged DataFrame (after optional multiple-match filtering) with an added
        column `Match_Status` where 1 indicates `id_a == id_b`, else 0.
    multiple matches before filtering when
        `keep_multiple` is False; otherwise None.
    """
    left = add_subscript('a', matchkey)
    right = add_subscript('b', matchkey)

    linked = dfA.merge(dfB, left_on=left, right_on=right, how='inner')

    multiple_matches = None
    if keep_multiple is False and not linked.empty:
        linked['multi_match'] = (
            linked['id_a'].map(linked['id_a'].value_counts() > 1) |
            linked['id_b'].map(linked['id_b'].value_counts() > 1)
        )
        multiple_matches = int(linked['multi_match'].sum())
        linked = linked[~linked['multi_match']].drop(columns='multi_match')

    if output_path is not None:
        os.makedirs(output_path, exist_ok=True)
        key_name = "_".join(matchkey)
        filename = f"linked_{key_name}.csv"
        full_path = os.path.join(output_path, filename)

        linked.convert_dtypes().to_csv(full_path, index=False, encoding="utf-8-sig")

    return linked, multiple_matches, left, right


### Compute residuals function
This function picks out all the records that do not match from the gold standard data unique identifier variables (id_a and id_b) and labels them as zero in the *match_status* variable and one when the records match. It then labels the links as *true_positive* and the nonlinks as *false_positive*. 

<div class="alert alert-block alert-danger">
 Note that calculating true positives and true negatives is possible for the whole dataset as this is a dummy dataset and you know the true links. When linking two different datasets and you do not know the true links, calculating True positives and false positives can be done in various different ways. Some examples are given below:<br>
Clerical review<br>
Using trusted, high confidence match keys (e.g. indidivual identifier, dob, name, etc...) as pseudo ground truth<br>
Probabilistic linkage and fellegi-sunter theory to estimate the ground truth 
</div>

To compute the residuals, an outer merge is done. This results in a dataset with all the records (those that match and those that do not match) named residuals.

This is then divided into two residuals for dfA and dfB:

<div class="alert alert-block alert-warning">
residualsA = (<br>
        residuals[<br>
            residuals['id_a'].notnull() &<br>
            ~residuals['id_a'].isin(linked['id_a'])<br>
        ][dfA.columns.values]<br>
        .drop_duplicates()<br>
    )
</div>

This code means that if records are in residuals and id_a are not null *residuals['id_a'].notnull()*, and not in linked *~residuals['id_a'].isin(linked['id_a'])* add them to residualsA and drop duplicates.

<div class="alert alert-block alert-warning">
unmatched = len(residualsA) + len(residualsB
</div>

This gives you the number of unmatched records in both datasets.

In [36]:
# computer residuals function
def compute_residuals(dfA, dfB, linked, matchkey, left, right):
    """
    Return rows in dfA and dfB whose IDs are NOT present in `linked` (post-link residuals). 
    
    Parameters
    ----------
    dfA, dfB : pd.DataFrame
        Source dataframes.
    linked : pd.DataFrame
        The successfully linked pairs, containing columns `id_a` and `id_b`.
    matchkeys : str
        Base column name used for linking (present as subscripted columns in dfA/dfB).
    match_status : bool
        Notifies if match has been made between two datasets.
    true_positives : int
        Number of true positives.
    false_positices : int
        Number of false positives.
    residuals: pd.Dataframe
        Residuals of matching dfA and dfB created by doing an outer join. 
    residualsA : pd.Dataframe
        residuals from dfA.
    residualsB : pd.Dataframe
        residuals from dfB.
    unmatched : pd.Dataframe
        unmatched residualsB
    id_a, id_b : str
        ID column names for A and B.

    Returns
    -------
    residualsA, residualsB, unmatchedB : (pd.DataFrame, pd.DataFrame, int)
        Unmatched rows for A and B; unmatchedB is the count of B residuals.
    """
    linked['Match_Status'] = np.where(linked['id_a'] == linked['id_b'], 1, 0)
    
    true_positives = len(linked[linked['Match_Status'] == 1])
    false_positives = len(linked[linked['Match_Status'] == 0])

    residuals = dfA.merge(dfB, left_on=left, right_on=right, how='outer')

    residualsA = (
        residuals[
            residuals['id_a'].notnull() &
            ~residuals['id_a'].isin(linked['id_a'])
        ][dfA.columns.values]
        .drop_duplicates()
    )

    residualsB = (
        residuals[
            residuals['id_b'].notnull() &
            ~residuals['id_b'].isin(linked['id_b'])
        ][dfB.columns.values]
        .drop_duplicates()
    )

    unmatched = len(residualsB)

    return residualsA, residualsB, (true_positives, false_positives, unmatched)


### Matching metrics and descriptors functions
The below are the matching metrics functions. These create match rate, precision and recall. 

A quick reminder of what all of these mean:
* update_match_rate: descriptor match numbers betweent the two datasets. Remember that this is not a quality metric! There are two match rate functions (print_match_rate and updated_match_rate) to ensure any results with updated matchkeys update the match rate.
* get_precision: Proportion of all links made, that are true matches.
* get_recall: Proportion of all true matches that were correctly linked.

In [37]:
# Update_match_rate function
def update_match_rate(true_positives, false_positives, link_status):
    """
    Update the number of true positives and false positives.

    Parameters
    ----------
    true_positives : int
        Current number of true positives.
    false_positives : int
        Current number of false positives.
    link_status : Tuple[int, int]
        A tuple containing (true_positive_increment, false_positive_increment).

    Returns
    -------
    Tuple[int, int]
        Updated (true_positives, false_positives).

    """
    true_positives += link_status[0]
    false_positives += link_status[1]
    return(true_positives, false_positives)

# get_precision function
def get_precision(true_positives, false_positives):
    """
    Calculate precision as a percentage rounded to 2 decimal places.

    Parameters
    ----------
    true_positives : int
        Number of true positives.
    false_positives : int
        Number of false positives.

    Returns
    -------
    float
        Precision percentage (0–100), rounded to 2 decimal places.
    """

    return round(true_positives/(true_positives+false_positives)*100, 2)

# get_recall function
def get_recall(true_positives, false_negatives):
    
    """
    Calculate recall as a percentage rounded to 2 decimal places.

    Parameters
    ----------
    true_positives : int
        Number of true positives.
    false_negatives : int
        Number of false negatives.

    Returns
    -------
    float
        Recall percentage (0–100), rounded to 2 decimal places.
    """
    return round(true_positives/(true_positives+false_negatives)*100, 2)

### Print match rate function
This function prints the output so that you can evaluate the strength of the match keys. 

In [38]:
# Print match rate function
def print_match_rate(final, exact, matchkey, multiple_matches, true_positives,
                     false_positives, unmatched, false_negatives):
    """
    Print details of the type of matching and counts of true/false positives, 
    false negatives, and unmatched cases.

    Parameters
    ----------
    final : bool
        Whether this is the overall/final match rate.
    exact : bool
        Whether it is exact matching.
    matchkey : List[str]
        List of matchkey fields used (printed only when exact=False and final=False).
    multiple_matches : Optional[int]
        Number of multiple matches, if applicable.
    true_positives : int
        Count of true positives.
    false_positives : int
        Count of false positives.
    unmatched : Optional[int]
        Number of unmatched records, if applicable.
    false_negatives : Optional[int]
        Count of false negatives, if applicable.
    """
    if final:
        print('Overall match rate')
    elif exact:
        print('Exact matching')
    else:
        print('Matchkey = ' + ', '.join(matchkey))
        if multiple_matches != None:
            print('Multiple matches = ' + str(multiple_matches))
    print('True positives = ' + str(true_positives))
    print('False positives = ' + str(false_positives))
    if unmatched != None:
        print('Unmatched = ' + str(unmatched))
    if false_negatives != None:
        print('False negatives = ' + str(false_negatives))
    print('')

### Wrapper functions 
Is_exact_key and rule_based_matching function wraps up the already present functions and gives you the desired results. 

In [39]:
def is_exact_key(dfA, left_cols):
    """
    Check whether a given list of columns exactly matches the full column order of a DataFrame.

    This function performs a strict equality check between:
      - `left_cols`: a list of column names (typically the left-hand matchkey),
      - the complete list of column names in `dfA`, in order.

    It returns True only if:
      left_cols == list(dfA.columns)

    Parameters
    ----------
    dfA : pandas.DataFrame
        The DataFrame whose column structure is being compared.

    left_cols : list of str
        A list of column names to test for exact equality with `dfA`'s columns.

    Returns
    -------
    bool
        True if `left_cols` exactly matches (same columns, same order) the columns
        of `dfA`; otherwise False.
    """

    return left_cols == dfA.columns.values.tolist()

def rule_based_matching(dfA, dfB, matchkey, keep_multiple, export_path=None):
    """
    Perform one pass of deterministic rule-based matching using a specified matchkey.

    This function:
      1. Calls `merge_on_matchkey` to link dfA and dfB on the subscripted versions
         of `matchkey`, optionally removing multi-matches depending on
         `keep_multiple`.
      2. Computes residual (unmatched) records for both DataFrames via
         `compute_residuals`.
      3. Returns residuals, statistics, and metadata needed for iterative linkage.

    Parameters
    ----------
    dfA : pandas.DataFrame
        The first dataset being matched. Must contain columns corresponding to
        `add_subscript('a', matchkey)` and an identifier column `id_a`.

    dfB : pandas.DataFrame
        The second dataset being matched. Must contain columns corresponding to
        `add_subscript('b', matchkey)` and an identifier column `id_b`.

    matchkey : list[str] or str
        The base variable(s) used to construct the left and right merge keys
        using `add_subscript`. For example, ['firstname','yob'] → 
        ['firstname_a','yob_a'] and ['firstname_b','yob_b'].

    keep_multiple : bool
        Whether to retain rows where an `id_a` or `id_b` matches multiple records
        in the merged dataset. If False, multi-matches are detected and removed.

    Returns
    -------
    residualsA : pandas.DataFrame
        Records from dfA not matched under the given matchkey.

    residualsB : pandas.DataFrame
        Records from dfB not matched under the given matchkey.

    stats : dict
        A dictionary of statistics returned by `compute_residuals` describing
        match counts, coverage, and diagnostics.

    multiple_matches : int or None
        The number of multi-matching rows removed (if `keep_multiple=False`);
        otherwise None.

    left : list[str] or str
        The left-hand merge key(s) used for matching, returned by
        `merge_on_matchkey`, useful for logging or debugging.

    """
    linked, multiple_matches, left, right = rule_based_linkage_via_matchkeys(dfA, dfB, matchkey, keep_multiple, output_path=export_path)
    residualsA, residualsB, stats = compute_residuals(dfA, dfB, linked, matchkey, left, right)
    return residualsA, residualsB, stats, multiple_matches, left, linked


### Call rule_based_matching, is exact and print match rate functions
The below code calls the functions and runs the linkage strategy first on any exact matching records. First it creates false_neg, true_pos and false_pos. It then runs the exact matching using matchkeys linkage function by running the rule based matching wrapper. Then it runs the is exact function and finally it calls the print match rate function to show the metrics for this stage of linkage. 

In [40]:
# Create false negatives, true_pos, false_pos before run matchkeys and create all_linked empty
# dataset ready for the exact matches
false_neg = len(dfB)
true_pos, false_pos = 0, 0
all_linked = []

In [41]:
#Run first round of rule-based matching using matchkeys
curA, curB = dfA, dfB
curA, curB, (true_pos, false_pos, unmatched), multiple_matches, left, linked_exact = rule_based_matching(
    curA, curB, remove_subscript(curA.columns.values), keep_multiple=False
)
exact = is_exact_key(curA, left)  
print_match_rate(False, exact, [], multiple_matches, true_pos, false_pos, unmatched, None)

linked_exact = linked_exact.copy()
linked_exact["matched_on"] = "EXACT"
all_linked.append(linked_exact)


Exact matching
True positives = 4474
False positives = 0
Unmatched = 2960



The above are some matching metrics after performing exact matching. 

Remember: true positives and false positives are computed here using the *id_a* and *id_b* variables. There are 13420 remaining unmatched records that you will increase the match rate for by using the matchkeys and amending them.

### Call rule based matching, print match rate, update match rate
This next section runs rule-based matching with matchkeys on the *remaining unmatched records*. It then calls print match rate, update match rate,  The metrics below show how many faslse positives and true positives against each matchkey with the final output showing the overall match rate with the overall prevision and recall. 

Running this section also creates a file containing matches with which matchkey the records were matched on.

In [42]:
# Run remaining functions
for mk in matchkeys:
    
    prevA = curA.copy()
    prevB = curB.copy()
    
    pre_cols = curA.columns.values.tolist()  
    curA, curB, (TP, FP, unmatched), multiple_matches, left, linked_step= rule_based_matching(
        curA, curB, mk, keep_multiple
    )

    print_match_rate(
        final=False,
        exact=False,
        matchkey=mk,
        multiple_matches=multiple_matches,
        true_positives=TP,
        false_positives=FP,
        unmatched=unmatched,
        false_negatives=None
    )

    true_pos, false_pos = update_match_rate(true_pos, false_pos, (TP, FP))

    
    if not linked_step.empty:
        linked_step = linked_step.copy()
        linked_step["matched_on"] = ",".join(mk)
        all_linked.append(linked_step)

false_neg = len(dfB) - true_pos

print_match_rate(True, False, None, None, true_pos, false_pos, None, false_neg)
print('Precision = ' + str(get_precision(true_pos, false_pos)) + '%')
print('Recall = ' + str(get_recall(true_pos, false_neg)) + '%')

if all_linked:
    df_matched_final = pd.concat(all_linked, ignore_index=True)
else:
    df_matched_final = pd.DataFrame(columns=dfA.columns.tolist() + dfB.columns.tolist() + ["matched_on"])

os.makedirs(filepath, exist_ok=True)

df_matched_final = df_matched_final.convert_dtypes()

final_matched_path = os.path.join(filepath, "FINAL_matched_records.csv")
df_matched_final.to_csv(final_matched_path, index=False, encoding="utf-8-sig")

print("Export complete:")
print(" - Combined matched records:", final_matched_path, f"({len(df_matched_final):,} rows)")


Matchkey = first1, middle1, sur1, postcode
Multiple matches = 54
True positives = 305
False positives = 96
Unmatched = 2559

Matchkey = yearbirth, sex, monthbirth, daybirth
Multiple matches = 50740
True positives = 0
False positives = 0
Unmatched = 2559

Overall match rate
True positives = 4779
False positives = 96
False negatives = 2655

Precision = 98.03%
Recall = 64.29%
Export complete:
 - Combined matched records: d:\repos\analysis-for-action\learning_resources\data_analysis\data_preparation\data_linkage\script_1\FINAL_matched_records.csv (4,875 rows)


### Outputs
These are the outputs from the matchkeys and will change as you amend said matchkeys. Each block of code outlines which matchkey it is related to after *Matchkey =*. It then outlines the number of multiple matches. Remember that in the first instance, although this output tells you how many multiple matches there are, multiple_matches are false in matchkeys.txt file and therefore not used therefore only one to one matches are performed. You can change this yourself and see how the outputs change accordingly if you amend *multiple_matches* to *True* in the matchkeys.txt file. 

After the number of multiple_matches are outlined, the number of true_positives and false_positives are outlined per matchkey. The final output block entitled *overall match rate* are the number of true positives and false positives from exact matching *AND* matchkeys (note that this is higher than the first block of outputs from only exact matching so using the matchkeys hsa increased the number of matches This output block also gives the total number of false negatives.

The output then produces precision and recall metrics along with a confirmation that the export is complete and where it is saved.

## **Matchkey solutions for exercise**

Remember that the thresholds you will be working towards are dependent on what your users are requesting. 
As a reminder precision means 'of all the record pairs that are found matches, how many are correct matches?' So high precision is rarely linking records that should not be linked whereas low precision is linking too many incorrect pairs. Recall measures how many of the true matching pairs are successfully linked. This means that high recall is missing few true links and low recall is missing many true links. 

For high precision (93.95%) and recall (86.27%) you can find an example below.

first1, middle1, sur1, postcode<br>
dob, sex<br>
first1, middle1, sur1, pcarea<br>
dob, sex, initials<br>
shortfirst1, shortmiddle1, shortsur1, postcode<br>
daybirth, monthbirth, yearbirth<br>

Starting with rigid matchkeys (full name, postcode, dob and sex), followed by relaxed matchkeys increases the match rate. 

For an example of lower precision (75%) and higher recall (86%) change the multiple matches to true in the matchkeys document with the below matchkeys.

shortfirst1, shortmiddle1, shortsur1, pcarea<br>
initials, daybirth, yearbirth, monthbirth<br>
initials, pcdistrict, monthbirth, yearbirth<br>

As expected, these much more relaxed matchkeys (as well as accepting mutliple matches) decreases the number of correct matches of the matches made (precision) and increases the number of true matching pairs linked (recall). 